# PHASE 5 — ADVANCED WORKFLOWS & PRODUCTION


# Day 26 — Cross Validation


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Explain why `train_test_split` can be dangerous and misleading.
- Understand the mechanics of **K-Fold Cross Validation**.
- Use `cross_val_score` to rigorously evaluate a model's true performance.
- Interpret the variance (standard deviation) of cross-validation scores to detect unstable models.


## 2. Prerequisites
- Day 3 (Train / Test Split).
- Day 12 & 13 (Decision Trees & Random Forests).


## 3. Concept: The "Luck of the Draw" Problem
Since Day 3, we have relied entirely on `train_test_split` to divide our data. We usually set `random_state=42` to ensure the split is the same every time.

But what if `random_state=42` happens to put all the "easy" rows into the test set? Your model will report an Accuracy of 95%. You deploy it to production, and it fails miserably. 
What if it puts all the "hard" outliers into the test set? The Accuracy reports 60%, and you throw away a perfectly good model.

Evaluating a model based on a single random slice of data is gambling. It is highly susceptible to the "Luck of the Draw".


## 4. Concept: K-Fold Cross Validation
**Cross Validation (CV)** mathematically eliminates this gamble.
Instead of splitting the data once, **K-Fold CV** does the following:
1. Cuts the entire dataset into $K$ equal-sized chunks (Folds). Let's say $K=5$.
2. It trains the model on Folds 1, 2, 3, and 4. It tests on Fold 5. It saves the score.
3. It throws the model away. It trains a brand new model on Folds 1, 2, 3, and 5. It tests on Fold 4. It saves the score.
4. It repeats this until every single Fold has been used as the Test set exactly once.

You now have 5 scores. You calculate the **Mean** (the true performance) and the **Standard Deviation** (the stability/variance) of those scores.


## 5. Scikit-learn API
```python
from sklearn.model_selection import cross_val_score
# cv=5 means 5-Fold Cross Validation
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
```


## 6. Simple Example: The Danger of train_test_split
Let's generate a complex dataset. We will train a `DecisionTreeClassifier` and evaluate it using 5 different `random_state` values in `train_test_split`. Watch how wildly the accuracy fluctuates!


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Generate Dataset
X, y = make_classification(n_samples=500, n_features=10, n_informative=5, random_state=42)

model = DecisionTreeClassifier(random_state=1)

# 2. Loop through 5 different random seeds for train_test_split
print('Accuracies across different single splits:')
for seed in [10, 42, 99, 123, 777]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f'Seed {seed}: {acc * 100:.1f}%')


## 7. Code Walkthrough
- We ran the exact same Decision Tree on the exact same dataset 5 times.
- The only thing that changed was which 20% of rows were randomly selected for the Test set.
- The accuracy swung wildly from ~75% to ~90%. If you used Seed 99, you'd think your model was amazing. If you used Seed 10, you'd think it was terrible. This is unacceptable for production.


## 8. Experiment: Using K-Fold CV
Let's fix this using `cross_val_score(cv=5)`. This will automatically slice the data into 5 equal folds, train 5 separate models, and return the 5 scores.


In [ ]:
from sklearn.model_selection import cross_val_score

# 1. Run 5-Fold Cross Validation (Notice we pass the ENTIRE X and y, not X_train!)
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('The 5 individual Fold scores:')
print(np.round(cv_scores, 3))

print('\nThe True Evaluation:')
print(f'Mean Accuracy: {cv_scores.mean() * 100:.1f}%')
print(f'Standard Dev:  ±{cv_scores.std() * 100:.1f}%')


> This tells us the absolute truth: The model is actually an 82% accurate model, and its performance can swing up or down by about 3.5% depending on what data it sees. We now have total confidence in our evaluation.


## 9. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=1)
rf_scores = cross_val_score(rf_model, X, y, cv=5, scoring='accuracy')


> **Question:** We know Random Forests average out the variance of single Decision Trees. What do you expect to happen to the `Mean Accuracy` and the `Standard Dev` when we run 5-Fold CV on the Random Forest compared to the single Tree?

**Think before running the next cell!**


In [ ]:
print('Random Forest True Evaluation:')
print(f'Mean Accuracy: {rf_scores.mean() * 100:.1f}%')
print(f'Standard Dev:  ±{rf_scores.std() * 100:.1f}%')
print('\nWhy? The Mean Accuracy went up drastically because Ensembles are mathematically superior.')
print('The Standard Dev went down drastically! The Random Forest is incredibly STABLE. It doesn\'t care which data fold you test it on, it almost always scores ~92%.')


## 10. Coding Exercise
Try running a **10-Fold** Cross Validation on the `RandomForestClassifier`. 
(Change `cv=10`). 
Print out the Mean and the Standard Deviation.


In [ ]:
# YOUR CODE HERE
rf_10_scores = cross_val_score(rf_model, X, y, cv=10, scoring='accuracy')
print(f'10-Fold Mean Accuracy: {rf_10_scores.mean() * 100:.1f}%')
print(f'10-Fold Standard Dev:  ±{rf_10_scores.std() * 100:.1f}%')


## 11. Debugging Challenge
A data scientist builds a complex Pipeline with `StandardScaler` and `SVC(kernel='rbf')`. 
They do `X_scaled = scaler.fit_transform(X)`, and then run `cross_val_score(model, X_scaled, y, cv=5)`. This is a massive mathematical error known as Data Leakage. Why?


In [ ]:
# Conceptual Bug
print('Error: Data Leakage in Cross Validation.')
print('Why? You scaled the ENTIRE dataset before slicing it into folds.')
print('The scaler looked at Fold 5 (the future test set) to calculate its Mean and Std. It then used that information to scale Fold 1, 2, 3, and 4.')
print('The training folds have technically "seen" the mathematical properties of the test fold! Your CV scores will be artificially inflated.')


> **Rule:** You must NEVER manually scale data before running `cross_val_score`. You must pass the **Pipeline** itself into `cross_val_score`. Scikit-learn is smart enough to properly run `.fit_transform()` only on the training folds, and `.transform()` on the test fold during every single iteration!


## 12. Model Evaluation (Tradeoffs)
- **Why not use K-Fold CV for everything?** 
Because it is incredibly slow. If you have a massive neural network that takes 5 hours to train, running `cv=5` will take 25 hours. 
- We use `train_test_split` during rapid prototyping when we just want a quick, dirty answer.
- We use `cross_val_score` at the very end of the project to rigorously certify the model before putting it into Production.


## 13. Real-World Example
**Medical Diagnosis Models**: If you are building a model to detect lung cancer, and you get 95% accuracy using a single `train_test_split`, the FDA will laugh you out of the room. They will mandate 10-Fold Cross Validation. If your 10-Fold CV shows `Mean: 95%, Std: ± 15%`, your model will be rejected. A Std of 15% means the model is highly unstable and could arbitrarily drop to 80% accuracy depending on the hospital it is deployed in.


## 14. Mini Project
Let's do it the correct, leak-free way. Build a Pipeline containing `StandardScaler` and `SVC`. 
Run a 5-Fold Cross Validation on the Pipeline itself using the raw `X` and `y`. Print the Mean and Std!


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

# Pass the PIPELINE into cross_val_score, along with RAW X and y
safe_cv_scores = cross_val_score(svm_pipe, X, y, cv=5, scoring='accuracy')

print(f'Leak-Free SVM Pipeline Mean: {safe_cv_scores.mean() * 100:.1f}%')
print(f'Leak-Free SVM Pipeline Std:  ±{safe_cv_scores.std() * 100:.1f}%')


## 15. Common Mistakes
- **Scaling before CV**: The #1 cause of data leakage in advanced machine learning. Always put the Scaler in a Pipeline and pass the Pipeline to `cross_val_score`.
- **Choosing K too large**: Setting `cv=100` (Leave-One-Out CV) is mathematically interesting but will cause the training time to explode.
- **Only looking at the Mean**: Always look at the Standard Deviation. An unstable model is a dangerous model.


## 16. Interview Questions
- **Beginner**: Why is `train_test_split` alone considered risky for final evaluation? (Answer: Because it relies on a single random slice of data. It could get lucky or unlucky).
- **Intermediate**: Explain exactly how 5-Fold Cross Validation works. (Answer: It splits the data into 5 chunks. It trains on 4 chunks and tests on the 5th. It repeats this 5 times so every chunk is the test set exactly once. It averages the 5 scores).
- **Advanced**: How does using a Pipeline with `cross_val_score` prevent data leakage? (Answer: If you scale before CV, the scaler's math is influenced by the test folds. The Pipeline ensures the scaler only runs `.fit()` on the training folds inside each of the 5 iterations, perfectly simulating reality).


## 17. Knowledge Check
- What function runs K-Fold Cross Validation? (`cross_val_score`)
- If a CV array is `[0.90, 0.91, 0.89, 0.90]`, is this model stable? (Yes, the standard deviation is extremely low)


## 18. Summary
- **train_test_split** is fast for prototyping, but subject to random luck.
- **K-Fold Cross Validation** rigorously trains and tests the model on every single row of data.
- We evaluate the **Mean** (true accuracy) and **Standard Deviation** (stability).
- You must pass a **Pipeline** to `cross_val_score` to avoid scaling data leakage.
- Random Forests are vastly more stable (lower Std) than single Decision Trees.


## 19. Homework
Load the `load_wine` dataset. Run a 5-Fold Cross validation on a raw `LogisticRegression()` model. 
Then, build a Pipeline with `StandardScaler` and `LogisticRegression()`, and run 5-Fold CV on the Pipeline. 
Compare the Mean Accuracies. You should see a massive improvement when the scaling is properly executed within the folds!
